In [1]:
filtered_dir = "/home/slow_data/Air_Quality/filtered_envisoft_air_quality_weather_data.csv"

In [2]:
import pandas as pd

# Đọc dữ liệu gốc và chuyển Timestamp thành datetime
df = pd.read_csv('/home/slow_data/Air_Quality/envisoft_air_quality_weather_data.csv', parse_dates=['Timestamp'], dayfirst=True)

# Điều kiện lọc riêng cho từng trạm
condition_hcm = (
    (df['Name'] == 'HCM: Khu Liên cơ quan Bộ Tài Nguyên và Môi Trường - số 20 Đ. Lý Chính Thắng (KK)') &
    (df['Timestamp'] >= pd.Timestamp('2025-04-25'))
)

condition_quangbinh = (
    (df['Name'] == 'Quảng Bình: Khu kinh tế Hòn La (KK)') &
    (df['Timestamp'] >= pd.Timestamp('2025-04-22'))
)

# Các trạm còn lại giữ nguyên
condition_others = ~df['Name'].isin([
    'HCM: Khu Liên cơ quan Bộ Tài Nguyên và Môi Trường - số 20 Đ. Lý Chính Thắng (KK)',
    'Quảng Bình: Khu kinh tế Hòn La (KK)'
])

# Gộp điều kiện lọc và lọc dữ liệu
filtered_df = df[condition_hcm | condition_quangbinh | condition_others].copy()

# Định dạng lại cột Timestamp theo dạng dd/mm/yyyy HH:MM
filtered_df['Timestamp'] = filtered_df['Timestamp'].dt.strftime('%d/%m/%Y %H:%M')

# Ghi đè lại vào file
filtered_df.to_csv(filtered_dir, index=False)

print("✅ Đã lọc và lưu lại file air_quality_weather_data_filtered.csv với định dạng thời gian chuẩn.")


✅ Đã lọc và lưu lại file air_quality_weather_data_filtered.csv với định dạng thời gian chuẩn.


In [3]:
df.head()

,Timestamp,Name,Latitude,Longitude,AQI,PM2.5,PM10,CO,NO2,O3,SO2,Temperature,Humidity,Pressure,Wind Speed
0,2025-04-08 14:00:00,Hà Nội: 556 Nguyễn Văn Cừ (KK),21.0333,105.8500,134,134.333871,80.967371,8.601812,52.59890,8.319781,5.554680,27.96,71.0,1012.0,3.90
1,2025-04-08 14:00:00,HCM: Khu Liên cơ quan Bộ Tài Nguyên và Môi Trư...,10.7500,106.6667,91,90.676780,58.723927,NaN,7.41375,81.791625,77.385556,35.01,46.0,1009.0,1.54
2,2025-04-08 14:00:00,HCM: Đ. Lê Hữu Kiều - P. Bình Trưng Tây - Quận...,10.7500,106.6667,95,95.340935,59.552588,NaN,8.45665,NaN,16.505320,35.01,46.0,1009.0,1.54
3,2025-04-08 14:00:00,Long An: UBND Tp Tân An - 76 Hùng Vương - P.2 ...,10.5333,106.4167,89,88.681186,52.501889,29.512184,NaN,NaN,1.959320,29.45,53.0,1009.0,5.84
4,2025-04-08 14:00:00,Hà Nội: ĐHBK cổng Parabol đường Giải Phóng (KK),21.0245,105.8412,151,151.418019,72.311020,NaN,12.98290,9.472219,2.942000,28.00,70.0,1012.0,3.85


In [4]:
import pandas as pd

# Đọc dữ liệu
df = pd.read_csv(filtered_dir, dayfirst=True)

# Chuyển Timestamp về đúng định dạng
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')

# Các thông số cần kiểm tra
columns_to_check = ['PM2.5', 'PM10', 'CO', 'NO2', 'O3', 'SO2',
                    'Temperature', 'Humidity', 'Pressure', 'Wind Speed']

# Chuyển các cột về số
for col in columns_to_check:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Ngưỡng kiểm tra bất thường (vượt quá là vô lý)
thresholds = {
    'PM2.5': (0, 1000),        # µg/m³
    'PM10': (0, 1500),
    'CO': (0, 100),            # mg/m³
    'NO2': (0, 1000),
    'O3': (0, 1000),
    'SO2': (0, 1000),
    'Temperature': (-50, 60),  # độ C
    'Humidity': (0, 100),      # %
    'Pressure': (850, 1100),   # hPa
    'Wind Speed': (0, 60)      # m/s
}

# Lọc và in ra các giá trị bất thường
print("📌 Các giá trị vượt ngưỡng bất thường:\n")

for col, (min_val, max_val) in thresholds.items():
    outliers = df[(df[col] < min_val) | (df[col] > max_val)]
    if not outliers.empty:
        print(f"🔴 {col}: {len(outliers)} giá trị bất thường")
        print(outliers[['Timestamp', 'Name', col]], "\n")  # In ra 5 dòng đầu tiên
    else:
        print(f"✅ {col}: Không có giá trị bất thường")


📌 Các giá trị vượt ngưỡng bất thường:

✅ PM2.5: Không có giá trị bất thường
✅ PM10: Không có giá trị bất thường
🔴 CO: 64 giá trị bất thường
                Timestamp                                               Name  \
37573 2025-05-09 11:00:00  HCM: Đ. Lê Hữu Kiều - P. Bình Trưng Tây - Quận...   
37585 2025-05-09 12:00:00  HCM: Đ. Lê Hữu Kiều - P. Bình Trưng Tây - Quận...   
37596 2025-05-09 13:00:00  HCM: Đ. Lê Hữu Kiều - P. Bình Trưng Tây - Quận...   
37609 2025-05-09 14:00:00  HCM: Đ. Lê Hữu Kiều - P. Bình Trưng Tây - Quận...   
37622 2025-05-09 15:00:00  HCM: Đ. Lê Hữu Kiều - P. Bình Trưng Tây - Quận...   
...                   ...                                                ...   
68746                 NaT                Quảng Bình: Khu kinh tế Hòn La (KK)   
70283                 NaT                Quảng Bình: Khu kinh tế Hòn La (KK)   
70389                 NaT                Quảng Bình: Khu kinh tế Hòn La (KK)   
70411                 NaT                Quảng Bình: Khu kin

In [5]:
# df = pd.read_csv(filtered_dir, dayfirst=True)
# Check all rows with Timestamp = NaT
nat_rows = df[df['Timestamp'].isna()]
print(f"🔴 Rows with NaT Timestamp: {len(nat_rows)}")
if not nat_rows.empty:
    print(nat_rows[['Timestamp', 'Name']])
else:
    print("✅ No rows with NaT Timestamp")

🔴 Rows with NaT Timestamp: 51260
      Timestamp                                               Name
1091        NaT    Hà Nội: ĐHBK cổng Parabol đường Giải Phóng (KK)
1092        NaT                     Hà Nội: 556 Nguyễn Văn Cừ (KK)
1093        NaT  Hà Nội: Công viên Nhân Chính - Khuất Duy Tiến ...
1094        NaT  HCM: Đ. Lê Hữu Kiều - P. Bình Trưng Tây - Quận...
1095        NaT  Đà Nẵng: Khuôn viên trường ĐH sư phạm Đà Nẵng ...
...         ...                                                ...
84888       NaT  Bắc Giang: Khu liên cơ quan tỉnh Bắc Giang - P...
84889       NaT  Hà Nam: Công Viên Nam Cao - P.Quang Trung - TP...
84890       NaT  Long An: UBND Tp Tân An - 76 Hùng Vương - P.2 ...
84891       NaT  Bình Dương: số 593 Đại lộ Bình Dương, P. Hiệp ...
84892       NaT                Quảng Bình: Khu kinh tế Hòn La (KK)

[51260 rows x 2 columns]


In [6]:
import pandas as pd

df = pd.read_csv(filtered_dir, dayfirst=True)

new_locations = [
    ("Hà Nội: ĐHBK cổng Parabol đường Giải Phóng (KK)", 21.0052, 105.8418),
    ("Hà Nội: 556 Nguyễn Văn Cừ (KK)", 21.0491, 105.8831),
    ("Hà Nội: Công viên Nhân Chính - Khuất Duy Tiến (KK)", 21.0031, 105.7947),
    ("HCM: Khu Liên cơ quan Bộ Tài Nguyên và Môi Trường - số 20 Đ. Lý Chính Thắng (KK)", 10.7823, 106.6834),
    ("HCM: Đ. Lê Hữu Kiều - P. Bình Trưng Tây - Quận 2 (Ngã ba Lê Hữu Kiểu và Trương Văn Bang) (KK)", 10.7823, 106.7528),
    ("Đà Nẵng: Khuôn viên trường ĐH sư phạm Đà Nẵng (KK)", 16.0622, 108.1594),
    ("Thái nguyên: Đường Hùng Vương - Tp Thái Nguyên (KK)", 21.59315, 105.8431),
    ("Phú Thọ: đường Hùng Vương - Tp Việt Trì (KK)", 21.33847, 105.3633),
    ("Bắc Giang: Khu liên cơ quan tỉnh Bắc Giang - P. Ngô Quyền - TP. Bắc Giang (KK)", 21.3015, 106.22603),
    ("Hà Nam: Công Viên Nam Cao - P.Quang Trung - TP. Phủ Lý (KK)", 20.536, 105.9165),
    ("Long An: UBND Tp Tân An - 76 Hùng Vương - P.2 (KK)", 10.5391, 106.4045),
    ("Bình Dương: số 593 Đại lộ Bình Dương, P. Hiệp Thành (KK)", 10.9923, 106.6577),
    ("Quảng Bình: Khu kinh tế Hòn La (KK)", 17.9329, 106.4966)
]

# Tạo dict để tìm nhanh
location_dict = {name: (lat, lon) for name, lat, lon in new_locations}

# Hàm tìm và cập nhật toạ độ
def update_coordinates(row):
    for loc_name in location_dict:
        if loc_name in row["Name"]:
            lat, lon = location_dict[loc_name]
            row["Latitude"] = lat
            row["Longitude"] = lon
            break
    return row

# Áp dụng cập nhật cho từng dòng
df = df.apply(update_coordinates, axis=1)

df.to_csv(filtered_dir, index=False)

In [7]:
import pandas as pd

# Đọc file CSV gốc
df = pd.read_csv(filtered_dir, dayfirst=True)


# Thêm cột 'Source' vào đầu tiên với giá trị 'gov'
df.insert(0, 'Source', 'gov')

# Lưu lại file mới (hoặc ghi đè nếu muốn)
df.to_csv(filtered_dir, index=False)

In [8]:
import pandas as pd

# Đọc file CSV gốc
df = pd.read_csv(filtered_dir, dayfirst=True)

# Tạo từ điển ánh xạ Name -> ID
name_to_id = {
    "Hà Nội: ĐHBK cổng Parabol đường Giải Phóng (KK)" :  31390903576425084107499649578,
    "Hà Nội: 556 Nguyễn Văn Cừ (KK)" : 28560877461938780203765592307,
    "Hà Nội: Công viên Nhân Chính - Khuất Duy Tiến (KK)" : 31390908889087377344742439468,
    "HCM: Khu Liên cơ quan Bộ Tài Nguyên và Môi Trường - số 20 Đ. Lý Chính Thắng (KK)" : 31390916083317566102523755051,
    "HCM: Đ. Lê Hữu Kiều - P. Bình Trưng Tây - Quận 2 (Ngã ba Lê Hữu Kiểu và Trương Văn Bang) (KK)" : 31390912357075263208060500522,
    "Đà Nẵng: Khuôn viên trường ĐH sư phạm Đà Nẵng (KK)" : 31388851800421997746903202346,
    "Thái nguyên: Đường Hùng Vương - Tp Thái Nguyên (KK)" : 29195707587706641566224751462,
    "Phú Thọ: đường Hùng Vương - Tp Việt Trì (KK)" : 28505268571336961948594948504,
    "Bắc Giang: Khu liên cơ quan tỉnh Bắc Giang - P. Ngô Quyền - TP. Bắc Giang (KK)" : 31387251434693138681789561386,
    "Hà Nam: Công Viên Nam Cao - P.Quang Trung - TP. Phủ Lý (KK)" : 31388883344354363840031242796,
    "Long An: UBND Tp Tân An - 76 Hùng Vương - P.2 (KK)" : 31390932574706768021562473002,
    "Bình Dương: số 593 Đại lộ Bình Dương, P. Hiệp Thành (KK)" : 31388839920718814259329251882,
    "Quảng Bình: Khu kinh tế Hòn La (KK)" : 29213751141295132066317063859,
}


# Tạo cột 'ID' từ Name
df.insert(1, 'ID', df['Name'].map(name_to_id))

# Lưu file kết quả
df.to_csv(filtered_dir, index=False)

In [9]:
import pandas as pd

# Đọc dữ liệu từ file gốc
df = pd.read_csv(filtered_dir)

# Loại bỏ dữ liệu của trạm Quảng Bình: KKT Hòn La (KK)
df_filtered = df[df['Name'] != 'Quảng Bình: KKT Hòn La (KK)']

# Ghi ra file mới (hoặc ghi đè lên file cũ nếu muốn)
df_filtered.to_csv(filtered_dir, index=False)



In [10]:
print(df_filtered['Name'].unique())


['Hà Nội: 556 Nguyễn Văn Cừ (KK)'
 'HCM: Đ. Lê Hữu Kiều - P. Bình Trưng Tây - Quận 2 (Ngã ba Lê Hữu Kiểu và Trương Văn Bang) (KK)'
 'Long An: UBND Tp Tân An - 76 Hùng Vương - P.2 (KK)'
 'Hà Nội: ĐHBK cổng Parabol đường Giải Phóng (KK)'
 'Hà Nội: Công viên Nhân Chính - Khuất Duy Tiến (KK)'
 'Đà Nẵng: Khuôn viên trường ĐH sư phạm Đà Nẵng (KK)'
 'Thái nguyên: Đường Hùng Vương - Tp Thái Nguyên (KK)'
 'Phú Thọ: đường Hùng Vương - Tp Việt Trì (KK)'
 'Bắc Giang: Khu liên cơ quan tỉnh Bắc Giang - P. Ngô Quyền - TP. Bắc Giang (KK)'
 'Hà Nam: Công Viên Nam Cao - P.Quang Trung - TP. Phủ Lý (KK)'
 'Bình Dương: số 593 Đại lộ Bình Dương, P. Hiệp Thành (KK)'
 'Quảng Bình: Khu kinh tế Hòn La (KK)'
 'HCM: Khu Liên cơ quan Bộ Tài Nguyên và Môi Trường - số 20 Đ. Lý Chính Thắng (KK)'
 'Hà Nội: Số 1 đường Giải Phóng - phường Bạch Mai - ĐHBK (KK)'
 'Hà Nội: Số 1 đường Giải Phóng - phường Bạch Mai (KK)'
 'HCM: Khu Liên cơ quan Bộ Tài Nguyên và Môi Trường - số 200 Đ. Lý Chính Thắng (KK)']


## Some change in name? Validate

In [12]:
import pandas as pd

# 1. Load the data and ensure Timestamp is a datetime object for validation
df = pd.read_csv(filtered_dir)

# Đảm bảo dữ liệu là chuỗi, xóa khoảng trắng thừa và parse với format chuẩn "08/04/2025 14:00"
df['Timestamp'] = pd.to_datetime(df['Timestamp'].astype(str).str.strip(), format='%d/%m/%Y %H:%M', errors='coerce')

# Kiểm tra xem có dòng nào bị lỗi format và biến thành NaT không
nat_count = df['Timestamp'].isna().sum()
if nat_count > 0:
    print(f"⚠️ Cảnh báo: Có {nat_count} dòng thời gian không đúng chuẩn '%d/%m/%Y %H:%M' và bị biến thành NaT.")
else:
    print("✅ Toàn bộ cột Timestamp đã được parse thành công, không có lỗi định dạng!")

# Define the suspicious groups to validate
groups_to_check = {
    "Group 1 (Hà Nội - Giải Phóng)": [
        'Hà Nội: ĐHBK cổng Parabol đường Giải Phóng (KK)',
        'Hà Nội: Số 1 đường Giải Phóng - phường Bạch Mai - ĐHBK (KK)',
        'Hà Nội: Số 1 đường Giải Phóng - phường Bạch Mai (KK)'
    ],
    "Group 2 (HCM - Lý Chính Thắng)": [
        'HCM: Khu Liên cơ quan Bộ Tài Nguyên và Môi Trường - số 20 Đ. Lý Chính Thắng (KK)',
        'HCM: Khu Liên cơ quan Bộ Tài Nguyên và Môi Trường - số 200 Đ. Lý Chính Thắng (KK)'
    ]
}

# --- VALIDATION STEP ---
print("--- VALIDATION: Time ranges for similar station names ---\n")
for group_name, stations in groups_to_check.items():
    print(f"{group_name}:")
    for station in stations:
        station_data = df[df['Name'] == station]
        if not station_data.empty:
            min_time = station_data['Timestamp'].min()
            max_time = station_data['Timestamp'].max()
            count = len(station_data)
            print(f"  - {station}:")
            print(f"      Records: {count} | From: {min_time} To: {max_time}")
        else:
            print(f"  - {station}: No data found.")
    print("-" * 60)

✅ Toàn bộ cột Timestamp đã được parse thành công, không có lỗi định dạng!
--- VALIDATION: Time ranges for similar station names ---

Group 1 (Hà Nội - Giải Phóng):
  - Hà Nội: ĐHBK cổng Parabol đường Giải Phóng (KK):
      Records: 6351 | From: 2025-04-08 14:00:00 To: 2026-02-06 10:00:00
  - Hà Nội: Số 1 đường Giải Phóng - phường Bạch Mai - ĐHBK (KK):
      Records: 402 | From: 2026-02-06 12:00:00 To: 2026-02-26 15:00:00
  - Hà Nội: Số 1 đường Giải Phóng - phường Bạch Mai (KK):
      Records: 394 | From: 2026-02-26 16:00:00 To: 2026-03-18 08:00:00
------------------------------------------------------------
Group 2 (HCM - Lý Chính Thắng):
  - HCM: Khu Liên cơ quan Bộ Tài Nguyên và Môi Trường - số 20 Đ. Lý Chính Thắng (KK):
      Records: 6581 | From: 2025-04-25 00:00:00 To: 2026-03-11 08:00:00
  - HCM: Khu Liên cơ quan Bộ Tài Nguyên và Môi Trường - số 200 Đ. Lý Chính Thắng (KK):
      Records: 133 | From: 2026-03-11 11:00:00 To: 2026-03-18 08:00:00
-------------------------------------

In [13]:
# --- MERGING STEP ---
# Create a mapping dictionary to replace variant names with your standard names
name_mapping = {
    'Hà Nội: Số 1 đường Giải Phóng - phường Bạch Mai - ĐHBK (KK)': 'Hà Nội: ĐHBK cổng Parabol đường Giải Phóng (KK)',
    'Hà Nội: Số 1 đường Giải Phóng - phường Bạch Mai (KK)': 'Hà Nội: ĐHBK cổng Parabol đường Giải Phóng (KK)',
    'HCM: Khu Liên cơ quan Bộ Tài Nguyên và Môi Trường - số 200 Đ. Lý Chính Thắng (KK)': 'HCM: Khu Liên cơ quan Bộ Tài Nguyên và Môi Trường - số 20 Đ. Lý Chính Thắng (KK)'
}

# Apply the replacement
df['Name'] = df['Name'].replace(name_mapping)

# Re-map the IDs to ensure the newly merged rows get the correct ID
# (Using the dictionary you defined in cell 8)
name_to_id = {
    "Hà Nội: ĐHBK cổng Parabol đường Giải Phóng (KK)" :  31390903576425084107499649578,
    "Hà Nội: 556 Nguyễn Văn Cừ (KK)" : 28560877461938780203765592307,
    "Hà Nội: Công viên Nhân Chính - Khuất Duy Tiến (KK)" : 31390908889087377344742439468,
    "HCM: Khu Liên cơ quan Bộ Tài Nguyên và Môi Trường - số 20 Đ. Lý Chính Thắng (KK)" : 31390916083317566102523755051,
    "HCM: Đ. Lê Hữu Kiều - P. Bình Trưng Tây - Quận 2 (Ngã ba Lê Hữu Kiểu và Trương Văn Bang) (KK)" : 31390912357075263208060500522,
    "Đà Nẵng: Khuôn viên trường ĐH sư phạm Đà Nẵng (KK)" : 31388851800421997746903202346,
    "Thái nguyên: Đường Hùng Vương - Tp Thái Nguyên (KK)" : 29195707587706641566224751462,
    "Phú Thọ: đường Hùng Vương - Tp Việt Trì (KK)" : 28505268571336961948594948504,
    "Bắc Giang: Khu liên cơ quan tỉnh Bắc Giang - P. Ngô Quyền - TP. Bắc Giang (KK)" : 31387251434693138681789561386,
    "Hà Nam: Công Viên Nam Cao - P.Quang Trung - TP. Phủ Lý (KK)" : 31388883344354363840031242796,
    "Long An: UBND Tp Tân An - 76 Hùng Vương - P.2 (KK)" : 31390932574706768021562473002,
    "Bình Dương: số 593 Đại lộ Bình Dương, P. Hiệp Thành (KK)" : 31388839920718814259329251882,
    "Quảng Bình: Khu kinh tế Hòn La (KK)" : 29213751141295132066317063859,
}
df['ID'] = df['Name'].map(name_to_id)

# Xóa các dòng trùng lặp (nếu có) phát sinh sau khi gộp tên trạm
df = df.drop_duplicates(subset=['Timestamp', 'ID'], keep='first')

# Convert Timestamp back to the string format used previously (if desired)
df['Timestamp'] = df['Timestamp'].dt.strftime('%d/%m/%Y %H:%M')

# Save the final consolidated file
df.to_csv(filtered_dir, index=False)

print("\n✅ Successfully merged duplicate station names and updated the IDs!")
print("Current unique stations:")
print(df['Name'].unique())


✅ Successfully merged duplicate station names and updated the IDs!
Current unique stations:
['Hà Nội: 556 Nguyễn Văn Cừ (KK)'
 'HCM: Đ. Lê Hữu Kiều - P. Bình Trưng Tây - Quận 2 (Ngã ba Lê Hữu Kiểu và Trương Văn Bang) (KK)'
 'Long An: UBND Tp Tân An - 76 Hùng Vương - P.2 (KK)'
 'Hà Nội: ĐHBK cổng Parabol đường Giải Phóng (KK)'
 'Hà Nội: Công viên Nhân Chính - Khuất Duy Tiến (KK)'
 'Đà Nẵng: Khuôn viên trường ĐH sư phạm Đà Nẵng (KK)'
 'Thái nguyên: Đường Hùng Vương - Tp Thái Nguyên (KK)'
 'Phú Thọ: đường Hùng Vương - Tp Việt Trì (KK)'
 'Bắc Giang: Khu liên cơ quan tỉnh Bắc Giang - P. Ngô Quyền - TP. Bắc Giang (KK)'
 'Hà Nam: Công Viên Nam Cao - P.Quang Trung - TP. Phủ Lý (KK)'
 'Bình Dương: số 593 Đại lộ Bình Dương, P. Hiệp Thành (KK)'
 'Quảng Bình: Khu kinh tế Hòn La (KK)'
 'HCM: Khu Liên cơ quan Bộ Tài Nguyên và Môi Trường - số 20 Đ. Lý Chính Thắng (KK)']
